In [1]:
try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
except: _numpy = "numpy"; _pil = "pillow"
!uv pip install -qqq \
    "torch>=2.8.0" "triton>=3.4.0" {_numpy} {_pil} torchvision bitsandbytes \
    unsloth "unsloth_zoo>=2026.4.6" transformers==5.5.0 torchcodec timm

In [2]:
from unsloth import FastModel
import torch

gemma4_models = [
    # Gemma-4 instruct models:
    "unsloth/gemma-4-E2B-it",
    "unsloth/gemma-4-E4B-it",
    "unsloth/gemma-4-31B-it",
    "unsloth/gemma-4-26B-A4B-it",
    # Gemma-4 base models:
    "unsloth/gemma-4-E2B",
    "unsloth/gemma-4-E4B",
    "unsloth/gemma-4-31B",
    "unsloth/gemma-4-26B-A4B",
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-4-31B-it",
    dtype = None, # None for auto detection
    max_seq_length = 8192, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
    device_map = "balanced", # Use 2x Tesla T4s on Kaggle
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.7: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.381 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/1188 [00:00<?, ?it/s]

In [3]:
from transformers import TextStreamer
# Helper function for inference
def do_gemma_4_inference(messages, max_new_tokens = 128):
    _ = model.generate(
        **tokenizer.apply_chat_template(
            messages,
            add_generation_prompt = True, # Must add for generation
            tokenize = True,
            return_dict = True,
            return_tensors = "pt",
        ).to("cuda"),
        max_new_tokens = max_new_tokens,
        use_cache = True,
        temperature = 1.0, top_p = 0.95, top_k = 64,
        streamer = TextStreamer(tokenizer, skip_prompt = True),
    )

Let's make a poem about sloths!

In [4]:
messages = [{
    "role": "user",
    "content": [{ "type" : "text",
                  "text" : "Write a poem about sloths in two lines." }]
}]
do_gemma_4_inference(messages)

A slow-motion dance in a canopy green,
The calmest creature that ever was seen.<turn|>


In [5]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False, # Turn off for just text!
    finetune_language_layers   = True,  # Should leave on!
    finetune_attention_modules = True,  # Attention good for GRPO
    finetune_mlp_modules       = True,  # Should leave on always!

    r = 8,           # Larger = higher accuracy, but might overfit
    lora_alpha = 8,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

In [6]:
dev = True

In [7]:
import os, json
from datasets import Dataset

def to_text(content):
    # content may be str, list of parts, dict, or None
    if content is None:
        return ""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for p in content:
            if isinstance(p, dict):
                parts.append(p.get("text") or p.get("content") or json.dumps(p))
            else:
                parts.append(str(p))
        return "\n".join(parts)
    if isinstance(content, dict):
        return content.get("text") or json.dumps(content)
    return str(content)

def load_trajectories(folder="trajectories"):
    convos = []
    for fname in sorted(os.listdir(folder)):
        if not fname.endswith(".jsonl"):
            continue
        msgs = []
        with open(os.path.join(folder, fname)) as f:
            for line in f:
                if not line.strip():
                    continue
                m = json.loads(line)
                msgs.append({"role": str(m["role"]), "content": to_text(m["content"])})
        if msgs:
            convos.append({"messages": msgs})
        if dev:
            convos = convos[:1000]
    return Dataset.from_list(convos)

dataset = load_trajectories("trajectories")
print(f"Loaded {len(dataset)} conversations")
print("Example:", dataset[0]["messages"][:2])

Loaded 1000 conversations
Example: [{'role': 'system', 'content': 'You are an autonomous coding agent. You have access to a shell and can run commands, read files, and edit code. Think step by step, investigate the problem, and make targeted fixes. Always verify your changes with `git diff` before finishing.'}, {'role': 'user', 'content': '"You are working on the PacketRusher codebase.\nThe repo implements a 5G core network testing tool that simulates user equipment and base stations for stress and validation testing. It is primarily written in Go with some C/C++ components for low-level networking and protocol handling. The project uses a Makefile for building and requires specific kernel modules and system dependencies.\n\nWorking directory is /repo.\nRequires custom kernel modules, root privileges, and specific Linux kernel headers; standard container environments may lack required system dependencies and SCTP support.\n\nThere is a data race from unsynchronized shared state access 

In [8]:
def normalize_roles(example):
    new_msgs = []
    for m in example["messages"]:
        role, content = m["role"], m["content"]
        if role == "tool":
            # Wrap tool output as a user-side observation
            content = f"<tool_output>\n{content}\n</tool_output>"
            role = "user"
        new_msgs.append({"role": role, "content": content})
    # Merge consecutive same-role messages (tool->user collapse can create these)
    merged = []
    for m in new_msgs:
        if merged and merged[-1]["role"] == m["role"]:
            merged[-1]["content"] += "\n" + m["content"]
        else:
            merged.append(dict(m))
    return {"messages": merged}

dataset = dataset.map(normalize_roles)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [9]:
len(dataset)

1000

In [10]:
def normalize_for_gemma(example):
    msgs = example["messages"]
    out = []
    pending_system = None

    for m in msgs:
        role, content = m["role"], m["content"]

        if role == "system":
            # Gemma-4 template doesn't accept system; prepend to next user turn
            pending_system = (pending_system + "\n\n" + content) if pending_system else content
            continue

        if role == "tool":
            content = f"<tool_output>\n{content}\n</tool_output>"
            role = "user"

        if role == "user" and pending_system:
            content = pending_system + "\n\n" + content
            pending_system = None

        # Merge into previous if same role
        if out and out[-1]["role"] == role:
            out[-1]["content"] += "\n" + content
        else:
            out.append({"role": role, "content": content})

    # If a stray system is still pending (no user turn came), drop it into a user turn
    if pending_system:
        out.insert(0, {"role": "user", "content": pending_system})

    # Must start with user
    if out and out[0]["role"] != "user":
        out.insert(0, {"role": "user", "content": ""})

    # Drop trailing user turn (no assistant to learn from)
    while out and out[-1]["role"] == "user":
        out.pop()

    return {"messages": out}

dataset = dataset.map(normalize_for_gemma, load_from_cache_file=False)
dataset = dataset.filter(lambda ex: len(ex["messages"]) >= 2, load_from_cache_file=False)
print(f"After normalization: {len(dataset)} conversations")
print("Sample roles:", [m["role"] for m in dataset[0]["messages"]])

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

After normalization: 1000 conversations
Sample roles: ['user', 'user', 'assistant', 'user', 'assistant', 'user', 'assistant', 'user', 'assistant', 'user', 'assistant', 'user', 'assistant', 'user', 'assistant', 'user', 'assistant', 'user', 'assistant', 'user', 'assistant', 'user', 'assistant', 'user', 'assistant', 'user', 'assistant', 'user', 'assistant', 'user', 'assistant', 'user', 'assistant', 'user', 'assistant', 'user', 'assistant', 'user', 'assistant', 'user', 'assistant', 'user', 'assistant', 'user', 'assistant', 'user', 'assistant', 'user', 'assistant']


In [11]:
from collections import Counter
first_roles = Counter()
issues = 0
for ex in dataset:
    roles = [m["role"] for m in ex["messages"]]
    first_roles[roles[0]] += 1
    for j in range(len(roles) - 1):
        if roles[j] == roles[j+1]:
            issues += 1
            break
print("First role:", first_roles)
print("Convos with consecutive duplicates:", issues)

First role: Counter({'user': 1000})
Convos with consecutive duplicates: 997


In [12]:
def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [
        tokenizer.apply_chat_template(
            convo, tokenize=False, add_generation_prompt=False
        ).removeprefix("<bos>")
        for convo in convos
    ]
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)
print(dataset[0]["text"][:1000])

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

<|turn>user
Agent submitted.<turn|>
<|turn>user
You are an autonomous coding agent. You have access to a shell and can run commands, read files, and edit code. Think step by step, investigate the problem, and make targeted fixes. Always verify your changes with `git diff` before finishing.

"You are working on the PacketRusher codebase.
The repo implements a 5G core network testing tool that simulates user equipment and base stations for stress and validation testing. It is primarily written in Go with some C/C++ components for low-level networking and protocol handling. The project uses a Makefile for building and requires specific kernel modules and system dependencies.

Working directory is /repo.
Requires custom kernel modules, root privileges, and specific Linux kernel headers; standard container environments may lack required system dependencies and SCTP support.

There is a data race from unsynchronized shared state access downstream of function Build in the test/aio5gc subsyste

In [13]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-4-thinking",
)

In [14]:
dataset[100]

{'messages': [{'role': 'user', 'content': 'Agent submitted.'},
  {'role': 'user',
   'content': 'You are an autonomous coding agent. You have access to a shell and can run commands, read files, and edit code. Think step by step, investigate the problem, and make targeted fixes. Always verify your changes with `git diff` before finishing.\n\n"You are working on the Open5GS codebase.\nThe repo implements a 5G core network system including components like AMF, SMF, and UPF in C, with some C++ modules. It uses the Meson build system and includes extensive protocol support for 5G networks.\n\nWorking directory is /repo.\nRequires Meson build system and several dependencies like libyaml, libmongoc, and libfdproto which may not be available in minimal containers\n\nThere is a heap buffer overflow due to incorrect size calculation downstream of function OpenAPI_ranking_criterion_convertToJSON in the lib/sbi/openapi subsystem.\nThe function is located at lib/sbi/openapi/model/ranking_criterion.

In [15]:
dataset[100]["text"]

'<|turn>user\nAgent submitted.<turn|>\n<|turn>user\nYou are an autonomous coding agent. You have access to a shell and can run commands, read files, and edit code. Think step by step, investigate the problem, and make targeted fixes. Always verify your changes with `git diff` before finishing.\n\n"You are working on the Open5GS codebase.\nThe repo implements a 5G core network system including components like AMF, SMF, and UPF in C, with some C++ modules. It uses the Meson build system and includes extensive protocol support for 5G networks.\n\nWorking directory is /repo.\nRequires Meson build system and several dependencies like libyaml, libmongoc, and libfdproto which may not be available in minimal containers\n\nThere is a heap buffer overflow due to incorrect size calculation downstream of function OpenAPI_ranking_criterion_convertToJSON in the lib/sbi/openapi subsystem.\nThe function is located at lib/sbi/openapi/model/ranking_criterion.c:31.\nPlease investigate and fix this issue.

<a name="Train"></a>
### Train the model
Now let's train our model. We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`.

In [16]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    eval_dataset = None, # Can set up evaluation!
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4, # Use GA to mimic batch size!
        warmup_steps = 5,
        # num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 60,
        learning_rate = 2e-4, # Reduce to 2e-5 for long training runs
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none", # Use TrackIO/WandB etc
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/1000 [00:00<?, ? examples/s]

In [17]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|turn>user\n",
    response_part = "<|turn>model\n",
)

Map (num_proc=16):   0%|          | 0/1000 [00:00<?, ? examples/s]

Filter (num_proc=16):   0%|          | 0/1000 [00:00<?, ? examples/s]

In [18]:
tokenizer.decode(trainer.train_dataset[100]["input_ids"])

'<|turn>user\nAgent submitted.<turn|>\n<|turn>user\nYou are an autonomous coding agent. You have access to a shell and can run commands, read files, and edit code. Think step by step, investigate the problem, and make targeted fixes. Always verify your changes with `git diff` before finishing.\n\n"You are working on the Open5GS codebase.\nThe repo implements a 5G core network system including components like AMF, SMF, and UPF in C, with some C++ modules. It uses the Meson build system and includes extensive protocol support for 5G networks.\n\nWorking directory is /repo.\nRequires Meson build system and several dependencies like libyaml, libmongoc, and libfdproto which may not be available in minimal containers\n\nThere is a heap buffer overflow due to incorrect size calculation downstream of function OpenAPI_ranking_criterion_convertToJSON in the lib/sbi/openapi subsystem.\nThe function is located at lib/sbi/openapi/model/ranking_criterion.c:31.\nPlease investigate and fix this issue.

In [19]:
tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[100]["labels"]]).replace(tokenizer.pad_token, " ")

"                                                                                                                                                                                                                                        I'll investigate and fix the heap buffer overflow issue in the OpenAPI_ranking_criterion_convertToJSON function. Let me start by examining the code.<turn|>\n                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  

In [20]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA A100-SXM4-40GB. Max memory = 39.381 GB.
18.16 GB of memory reserved.


In [ ]:
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,000 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 61,214,720 of 31,334,301,232 (0.20% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss


In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

<a name="Inference"></a>
### Inference
Let's run the model via Unsloth native inference! According to the `Gemma-4` team, the recommended settings for inference are `temperature = 1.0, top_p = 0.95, top_k = 64`

In [ ]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-4-thinking",
)
messages = [{
    "role": "user",
    "content": [{
        "type" : "text",
        "text" : "Continue the sequence: 1, 1, 2, 3, 5, 8,",
    }]
}]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
    tokenize = True,
    return_dict = True,
).to("cuda")
outputs = model.generate(
    **inputs,
    max_new_tokens = 64, # Increase for longer outputs!
    use_cache = True,
    # Recommended Gemma-4 settings!
    temperature = 1.0, top_p = 0.95, top_k = 64,
)
tokenizer.batch_decode(outputs)

 You can also use a `TextStreamer` for continuous inference - so you can see the generation token by token, instead of waiting the whole time!

In [ ]:
messages = [{
    "role": "user",
    "content": [{"type" : "text", "text" : "Why is the sky blue?",}]
}]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
    tokenize = True,
    return_dict = True,
).to("cuda")

from transformers import TextStreamer
_ = model.generate(
    **inputs,
    max_new_tokens = 64, # Increase for longer outputs!
    use_cache = True,
    # Recommended Gemma-4 settings!
    temperature = 1.0, top_p = 0.95, top_k = 64,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Hugging Face's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [ ]:
model.save_pretrained("gemma_4_lora")  # Local saving
tokenizer.save_pretrained("gemma_4_lora")
# model.push_to_hub("HF_ACCOUNT/gemma_4_lora", token = "YOUR_HF_TOKEN") # Online saving
# tokenizer.push_to_hub("HF_ACCOUNT/gemma_4_lora", token = "YOUR_HF_TOKEN") # Online saving

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [ ]:
if False:
    from unsloth import FastModel
    model, tokenizer = FastModel.from_pretrained(
        model_name = "gemma_4_lora", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = 2048,
        load_in_4bit = True,
    )

messages = [{
    "role": "user",
    "content": [{"type" : "text", "text" : "What is Gemma-4?",}]
}]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
    tokenize = True,
    return_dict = True,
).to("cuda")

from transformers import TextStreamer
_ = model.generate(
    **inputs,
    max_new_tokens = 128, # Increase for longer outputs!
    use_cache = True,
    # Recommended Gemma-4 settings!
    temperature = 1.0, top_p = 0.95, top_k = 64,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

### Saving to float16 for VLLM

We also support saving to `float16` directly for deployment! We save it in the folder `gemma-4-finetune`. Set `if False` to `if True` to let it run!

In [ ]:
if False: # Change to True to save finetune!
    model.save_pretrained_merged("gemma-4-finetune", tokenizer)

If you want to upload / push to your Hugging Face account, set `if False` to `if True` and add your Hugging Face token and upload location!

In [ ]:
if False: # Change to True to upload finetune
    model.push_to_hub_merged(
        "HF_ACCOUNT/gemma-4-finetune", tokenizer,
        token = "YOUR_HF_TOKEN"
    )

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now for all models! For now, you can convert easily to `Q8_0, F16 or BF16` precision. `Q4_K_M` for 4bit will come later!

In [ ]:
if False: # Change to True to save to GGUF
    model.save_pretrained_gguf(
        "gemma_4_finetune",
        tokenizer,
        quantization_method = "Q8_0", # For now only Q8_0, BF16, F16 supported
    )

Likewise, if you want to instead push to GGUF to your Hugging Face account, set `if False` to `if True` and add your Hugging Face token and upload location!

In [ ]:
if False: # Change to True to upload GGUF
    model.push_to_hub_gguf(
        "HF_ACCOUNT/gemma_4_finetune",
        tokenizer,
        quantization_method = "Q8_0", # Only Q8_0, BF16, F16 supported
        token = "YOUR_HF_TOKEN",
    )

Now, use the `gemma-4-finetune.gguf` file or `gemma-4-finetune-Q4_K_M.gguf` file in llama.cpp.

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other resources:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
4. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://unsloth.ai/docs/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️
</div>

  This notebook and all Unsloth notebooks are licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).